# Numerički integratori i grafovske neuronske mreže za simulaciju dinamike čestica
**Projekat: NANS (Numerički Algoritmi i Numerički Softver)**  

Ova prezentacija predstavlja projekat za ocenu na kursu NANS.

## 1. Motivacija i Uvod

Simulacija dinamike čestica predstavlja ključno područje u računarskoj fizici. Tradicionalno se koriste **numerički integratori** kako bi se rešile diferencijalne jednačine kretanja iterativno kroz vreme. Ovi algoritmi su fizički utemeljeni, ali pate od velike osetljivosti na vremenski korak (`dt`) i izuzetno su računarski zahtevni.

Glavni cilj projekta je upoređivanje klasičnog numeričkog pristupa sa pristupom mašinskog učenja pomoću **Grafovskih Neuronskih Mreža (GNN)**, koje mogu omogućiti bržu simulaciju kompleksne fizike, ali je njihova velika prednost mogućnost simuliranja fizike samo na osnovu prethodnog učenja interakcija iz podataka (isključivo posmatranjem, bez opisa sistema - black box).


## 2. NANS u Fokusu: Numerička Simulacija (WCSPH)

Kao osnovu projekta i primarni izvor fizički korektnih podataka, razvijena je sopstvena **WCSPH (Weakly Compressible Smoothed Particle Hydrodynamics)** simulacija.

### Matematički Model
Dinamika tečnosti aproksimira se metodom čestica čije interakcije definiše SPH kernel funkcija $W(r, h)$, gde je $h$ radijus uticaja. Centralne jednačine obuhvataju:

1. **Gustina:** $\rho_i = \sum_j m_j W(r_{ij}, h)$  
2. **Pritisak fluida (Taitova jednačina stišljive tečnosti):** $P_i = k \left( \left(\frac{\rho_i}{\rho_0}\right)^\gamma - 1 \right)$
3. **Sila pritiska (gradijent pritiska SPH):** $\mathbf{F}_i^{press} = -\sum_j m_j \left( \frac{P_i}{\rho_i^2} + \frac{P_j}{\rho_j^2} \right) \nabla W(r_{ij}, h)$
4. **Sila viskoznosti:** $\mathbf{F}_i^{visc} = \mu \sum_j m_j \frac{\mathbf{v}_j - \mathbf{v}_i}{\rho_j} \nabla^2 W(r_{ij}, h)$

### Numerički Integrator u Simulaciji
Ključno za veran proračun numeričke simulacije iz projektnog zadatka je izbor efikasnog numeričkog integratora. Iako su podržani **Forward Euler** i **Runge-Kutta 4 (RK4)**, za neophodnu energetsku stabilnost WCSPH sistema implementirana je varijanta **Symplectic Euler** integratora:

$$ \mathbf{v}_{t+1} = \mathbf{v}_t + \mathbf{a}_t \Delta t $$
$$ \mathbf{x}_{t+1} = \mathbf{x}_t + \mathbf{v}_{t+1} \Delta t $$

Simplektički Euler obezbeđuje očuvanje energije duž veoma dugačkih trajektorija simulacije fluida uz izrazito niske računske zahteve.


In [ ]:
# Učitavamo okruženje i relevantne simulacione biblioteke.
import sys
import os
import torch
import numpy as np
import yaml
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display

if os.getcwd().split('/')[-1] == 'presentation':
    os.chdir('..')
sys.path.append(os.getcwd())

from simulation import FluidSimulation
from models import GNNSimulator
from dataset import build_graph
from presentation.utils import create_simulation_video, create_rollout_video

In [ ]:
# Čista numerička simulacija bez mašinskog učenja

sim_test = FluidSimulation(
    num_particles=600,
    gravitational_constant=9.81,
    softening_length=0.015,
    stiffness=1000,
    exponent=7.0,
    viscosity=0.3
)

position_scale=0.5

np.random.seed(42)
sim_test.initialize_random_state(
    position_scale=position_scale,
    velocity_scale=0.0,
    mass_range=(1.0, 1.0),
    start=(1,1)
)

# Simulate
dt = 0.0005
total_time = 5.0
fps = 60
save_every = round((total_time/dt)/(total_time*fps))

history_positions, history_velocities, history_times = sim_test.simulate(
    total_time=total_time,
    dt=dt,
    save_every=save_every
)

video_numeric = create_simulation_video(
    sim=sim_test,
    history_positions=history_positions,
    num_particles=sim_test.num_particles,
    position_scale=position_scale,
    fps=fps,
    filename='presentation/nans_wcsph_sim.mp4'
)
display(video_numeric)


## 3. Skupovi podataka i trening (Dataset generation, treniranje i Fine-Tuning)

Učenje ovog modela se zasniva na pre-training/fine-tuning ideji:

1. **Pre-training:** Model započinje razumevanje sveta treniranjem na obimnom DeepMind-ovom **WaterDrop datasetu**. Time mreža stiče razumevanje ograničenja kontejnera i dinamika kapljica.
2. **Fine-Tuning:** Generisan je sopstveni **WCSPH fluid dataset**. Pre-trenirana mreža se putem fine-tuning transfer learning pristupa na doučila na našu varijantu numerike. Fine-tuning je trajao 10x kraće od pre-training faze i dobijeni su uporedivi rezultati kao i pre-training faza za vreme rollout-a.

**Ključna razlika:** DeepMind-ov WaterDrop dataset je simuliran MPM simulatorom, dok je dataset za fine-tuning simuliran WCSPH pristupom. Ovo su dva veoma različita pristupa i predstavljaju različite distribucije podataka.

## 4. Arhitektura GNN Modela (Encode - Process - Decode)

Podaci generisani WCSPH metodom moraju se pretvoriti u graf pre nego što se ubace u model. Fizičko stanje fluida posmatramo kroz dinamičan *Radius Graph*:
- Svaka čestica fluida i granica prostora je **čvor**. Interakcije čestica u izabranom prečniku dejstva sila (radijusu dejstva jezgra $h$) postaju **veze/grane (edges)**. Time se kreira ulazni graf u arhitekturu.
- **Encoder:** Prvo projektuje prethodne položaje i brzine čestica u latentni (skriveni) prostor dimenzionalnosti $128$ koristeći klasični Multilayer Perceptron.
- **Processor:** Serija InteractionNetwork modula koji vrše message passing; simuliraju razmenu sila sa susednim česticama.
- **Decoder:** Nakon niza iteracija po česticama grafa, skriveno stanje nazad prevodimo u vektor izvedenih predviđenih **ubrzanja** $\mathbf{a}$.

Dobijen izlaz se provlači kroz **Euler Integrator** za konačno predviđanje pozicije $\mathbf{x}_{t+1}$.


In [ ]:
# Inicijalizacija parametara i težina fine-tuned modela
CONFIG_PATH = 'configs/wcsph_transfer.yaml'
CHECKPOINT_PATH = 'checkpoints/best_model_wcsph.pt'

with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# ucitavanje tezina
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)

# Kreiranje instance modela
model = GNNSimulator.from_config(config).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

broj_parametara = sum(p.numel() for p in model.parameters())
print(f"GNN Arhitektura je inicijalizovana.")
print(f"Veličina modela: {broj_parametara:,} ukupno utreniranih parametara.")


## 5. Rezultati i Vizuelizacija GNN modela (Rollout)

Za evaluaciju rezultata izvršavamo takozvani **Rollout**. Rollout predstavlja autoregresivno predviđanje trajektorije čestica korišćenjem GNN modela. Zatim prikazujemo **MSE** tokom rollout-a.

Rollout se izvršava sa ~3000 čestica, što je 4 puta veći broj neko što je korišćeno za treniranje, da bi se pokazala sposobnost generalizacije modela.

In [ ]:
# Priprema domena i pre-generisanje Ground Truth numeričke simulacije
WATERDROP_BOUNDS = [[0.1, 0.9], [0.1, 0.9]]
WATERDROP_DT = 0.0025          # Effective frame-to-frame dt
INTEGRATION_DT = 0.0005        # Sub-step dt for WCSPH stability
SAVE_EVERY = 5                 # INTEGRATION_DT * SAVE_EVERY = WATERDROP_DT
SMOOTHING_LENGTH = 0.015       # = connectivity_radius
SEQUENCE_LENGTH = 1000         # Frames per trajectory
POSITION_SCALE = 0.8           # Container size: [0, 0.8], shifted by +0.1 → [0.1, 0.9]
OFFSET = 0.1                   # Coordinate shift to align with WaterDrop bounds
PARTICLE_TYPE_FLUID = 5        # DeepMind's encoding for fluid particles

# Random seed for this visualization (change for different trajectories)
VIZ_SEED = 123
rng = np.random.RandomState(VIZ_SEED)

num_particles = rng.randint(2000, 4001)

total_time = SEQUENCE_LENGTH * SAVE_EVERY * INTEGRATION_DT

sim = FluidSimulation(
    num_particles=num_particles,
    gravitational_constant=9.81,
    softening_length=SMOOTHING_LENGTH,
    integrator='symplectic_euler',
    position_scale=POSITION_SCALE,
    rest_density=1000.0,
    stiffness=1000.0,
    exponent=7.0,
    viscosity=0.3,
)

# Randomized initial drop center within margins
margin = 0.15 * POSITION_SCALE
cx = rng.uniform(margin, POSITION_SCALE - margin)
cy = rng.uniform(0.3 * POSITION_SCALE, POSITION_SCALE - margin)

sim.initialize_random_state(
    position_scale=POSITION_SCALE,
    velocity_scale=0.0,
    mass_range=(1.0, 1.0),
    start=(cx, cy),
)

print(f"Generating trajectory: {num_particles} particles, drop center=({cx:.3f}, {cy:.3f})")
positions, _, _ = sim.simulate(
    total_time=total_time,
    dt=INTEGRATION_DT,
    save_every=SAVE_EVERY,
    quiet=False
)

positions = (positions[:SEQUENCE_LENGTH + 1] + OFFSET).astype(np.float32)

gt_velocities = (positions[1:] - positions[:-1]).astype(np.float32)
gt_positions = positions[1:].astype(np.float32)

particle_type = np.full(num_particles, PARTICLE_TYPE_FLUID, dtype=np.int64)
mass = np.ones(num_particles, dtype=np.float32)

T, N, D = gt_positions.shape
print(f"Trajectory: {T} timesteps, {N} particles, {D}D")
print(f"Position range: [{gt_positions.min():.4f}, {gt_positions.max():.4f}]")

metadata = {
    'bounds': WATERDROP_BOUNDS,
    'default_connectivity_radius': SMOOTHING_LENGTH,
    'dt': WATERDROP_DT,
}
bounds = metadata['bounds']


In [ ]:
from tqdm.notebook import tqdm

# Predikcije GNN Simulatora - Autoregresivan Rollout

history_window = 5 # 5 sekvencijalnih koraka koje gleda GNN
pred_positions = np.zeros_like(gt_positions[:T])
pred_positions[:history_window + 1] = gt_positions[:history_window + 1].copy()

# Konverzija za Torch Device
particle_type_t = torch.from_numpy(particle_type).to(device)
mass_t = torch.from_numpy(mass).to(device)
dummy_acc_t = torch.zeros(num_particles, 2, dtype=torch.float32, device=device)

vel_history_t = torch.from_numpy(gt_velocities[:history_window]).to(device)
current_pos_t = torch.from_numpy(gt_positions[history_window]).to(device)
current_vel_t = torch.from_numpy(gt_velocities[history_window - 1]).to(device)

# Sekvencijalna Forward inferenca modela!
with torch.no_grad():
    for t in tqdm(range(history_window, T - 1), desc='Stanje Mreže', disable=True):
        vel_flat_t = vel_history_t.transpose(0, 1).reshape(num_particles, history_window * 2)
        
        # Oformljuje radijus graf na bazi susedstva čestica preko CPU/GPU-a
        graph = build_graph(
            positions=current_pos_t, velocities=vel_flat_t, particle_type=particle_type_t,
            masses=mass_t, target_acc=dummy_acc_t, connectivity_radius=SMOOTHING_LENGTH, bounds=WATERDROP_BOUNDS
        )
        
        predicted_acc_t = model(graph)
        new_vel_t = current_vel_t + predicted_acc_t
        new_pos_t = current_pos_t + new_vel_t
        
        pred_positions[t + 1] = new_pos_t.cpu().numpy()
        vel_history_t = torch.cat([vel_history_t[1:], new_vel_t.unsqueeze(0)], dim=0)
        current_pos_t = new_pos_t
        current_vel_t = new_vel_t

# Per-step position MSE
rollout_start = history_window + 1
step_mse = np.mean((pred_positions[rollout_start:] - gt_positions[rollout_start:])**2, axis=(1, 2))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(rollout_start, T), step_mse, color='#e74c3c', linewidth=1.5)
ax.set_xlabel('Timestep')
ax.set_ylabel('Position MSE')
ax.set_title('WCSPH Rollout Error Over Time')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Mean rollout MSE: {step_mse.mean():.6f}")
print(f"Final step MSE: {step_mse[-1]:.6f}")


In [ ]:
# Vizualizacija - Uporedni prikaz istinitih i predvidivih položaja čestica (Animacija)
video_rollout = create_rollout_video(
    gt_positions=gt_positions,
    pred_positions=pred_positions,
    num_particles=num_particles,
    T=T,
    sim_dt=WATERDROP_DT,
    bounds=WATERDROP_BOUNDS,
    target_fps=60,
    filename='presentation/nans_gnn_rollout.mp4'
)
display(video_rollout)


## 6. Zaključak projekta

- **Uspešna primena GNS modela**: Implementacija Graph Network-based Simulator-a (GNS) potvrdila je nalaze DeepMind-a da se složena dinamika fluida i čestica može efikasno modelovati korišćenjem grafovskih neuronskih mreža.

- **Generalizacija i skalabilnost**: Model je pokazao sposobnost generalizacije na sisteme sa različitim brojem čestica i početnim uslovima i sposobnost efikasnog transfer learning-a za simuliranje fizičkih sistema van distribucije.

- **Encoder-Processor-Decoder arhitektura**: Message passing kroz više slojeva procesora omogućilo je mreži da nauči dugolinijske interakcije i kompleksne fizičke zakone direktno iz podataka, bez eksplicitnog definisanja diferencijalnih jednačina.

- **Budući pravci**: Iako su rezultati impresivni, dalji razvoj bi se mogao fokusirati na poboljšanje stabilnosti simulacija na veoma dugim vremenskim intervalima i optimizaciju memorijske potrošnje za rad sa milionima čestica u realnom vremenu.
  
  - Dodatno zanimljivo je istražiti mogućnosti GNN arhitekture za pravljenje "generalist" modela koji bi mogli da simuliraju kompletno različite sisteme fizike, npr. isti model sposoban da simulira fluid, čvrsta tela, elektromagnetizam, itd.

